# **Amazon Delivery Time Prediction**
### **Data Cleaning**

## Objective

The objective of this notebook is to clean and validate raw delivery data before downstream exploratory analysis and machine learning workflows.

---

## Key Goals

- Load raw Bronze-layer delivery data
- Validate schema and dataset structure
- Detect missing values and duplicates
- Perform data cleaning operations
- Generate a reliable Silver-layer dataset

# **<u>Load Bronze Layer Dataset</u>**

In [4]:
from pyspark.sql import functions as F

df = spark.read.csv("abfss://f3654803-0ae2-46c6-b5e3-bf931741cfac@onelake.dfs.fabric.microsoft.com/3fed2a27-9ec6-4038-9e8a-f3e0b408ce78/Files/bronze/amazon_delivery.csv",
    header=True,
    inferSchema=True
)

display(df)

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 289d1ecc-446b-416f-9bac-d08f13b2f64b)

### Dataset Overview

This section explores dataset dimensions and schema structure.

In [5]:
print("Shape:", (df.count(), len(df.columns)))

df.printSchema()

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 7, Finished, Available, Finished, False)

Shape: (43739, 16)
root
 |-- Order_ID: string (nullable = true)
 |-- Agent_Age: integer (nullable = true)
 |-- Agent_Rating: double (nullable = true)
 |-- Store_Latitude: double (nullable = true)
 |-- Store_Longitude: double (nullable = true)
 |-- Drop_Latitude: double (nullable = true)
 |-- Drop_Longitude: double (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_Time: string (nullable = true)
 |-- Pickup_Time: timestamp (nullable = true)
 |-- Weather: string (nullable = true)
 |-- Traffic: string (nullable = true)
 |-- Vehicle: string (nullable = true)
 |-- Area: string (nullable = true)
 |-- Delivery_Time: integer (nullable = true)
 |-- Category: string (nullable = true)



Overview Insight

The raw dataset contains operational delivery information including traffic, weather, vehicle, geographic, and temporal delivery attributes.

### Missing Value Analysis

This section identifies missing values across dataset columns.

In [6]:
from pyspark.sql.functions import col, isnan, when, count

null_counts = df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

display(null_counts)

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40c83d15-5ed4-49a7-9030-76cdcacf0546)

Missing value analysis helps identify incomplete operational records that may negatively impact downstream analytics and model performance.

### Duplicate Record Analysis

In [7]:
print("Total Rows:", df.count())

print(
    "Duplicate Rows:",
    df.count() - df.dropDuplicates().count()
)

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 9, Finished, Available, Finished, False)

Total Rows: 43739
Duplicate Rows: 0


Insight

Duplicate delivery records can introduce analytical bias and reduce model reliability if not removed.

### Duplicate Removal

In [8]:
df = df.dropDuplicates()

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 10, Finished, Available, Finished, False)

Cleaning Action

Duplicate delivery records were removed to improve dataset consistency and analytical accuracy.

### Missing Value Handling

In [9]:
df = df.dropna()

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 11, Finished, Available, Finished, False)

 Cleaning Action

Rows containing incomplete operational records were removed to ensure reliable downstream feature engineering and machine learning workflows.

### Post-Cleaning Validation

In [10]:
print("Rows After Cleaning:", df.count())

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 12, Finished, Available, Finished, False)

Rows After Cleaning: 43685


Validation Insight

The cleaned dataset is now suitable for exploratory analysis and feature engineering workflows.

#  <u>**Silver Layer Storage**</u>

The cleaned dataset is stored in the Silver layer for downstream analytics and machine learning processing.

In [11]:
df.write.mode("overwrite").parquet(
    "abfss://f3654803-0ae2-46c6-b5e3-bf931741cfac@onelake.dfs.fabric.microsoft.com/3fed2a27-9ec6-4038-9e8a-f3e0b408ce78/Files/silver/cleaned_delivery_data"
)

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 13, Finished, Available, Finished, False)

### Silver Layer Validation

In [12]:
silver_df = spark.read.parquet(
    "Files/silver/cleaned_delivery_data"
)

display(silver_df)

StatementMeta(, b1002f50-0421-42a4-adbf-5ac807154940, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf3709ca-f699-4cbe-a703-2ad35a7bfbce)

Validation Insight

The Silver-layer dataset has been successfully generated and validated for downstream processing.

# **<u>Key Data Cleaning Insights</u>**

- Missing values and duplicate records were successfully identified and handled.
- Schema validation confirmed dataset consistency.
- The cleaned Silver-layer dataset provides a reliable foundation for downstream analytics.
- Data quality improvements reduce the risk of biased machine learning outcomes.

---

# <u>**Next Steps**</u>

The cleaned Silver-layer dataset will be used for:
- exploratory data analysis
- feature engineering
- machine learning model development
- operational analytics